# residual.ipynb

Audio-Flamingo-3 representation reinjection experiments with multiple seeds and bootstrap summaries.

Set the representation roots, split file, and output directories, then run the chosen reinjection setting.

Use the setup steps in the repository README before running this notebook.


In [ ]:
!pip install iterative-stratification


In [ ]:
# =========================================================
# DIRECT REPRESENTATION EXTRACTION PROBE
# MULTILABEL SUBTLE EMOTION DECODING
# =========================================================

import os
import ast
import json
import glob
import shutil
import random

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import (
    f1_score
)

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedShuffleSplit
)

from scipy.stats import (
    wilcoxon
)

import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

# =========================================================
# CONFIG
# =========================================================

CSV_PATH = (
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv"
)

REP_ROOT = (
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo"
)

LOCAL_REP_ROOT = (
    "/content/reps_local_flamingo"
)

SAVE_DIR = (
    "/content/drive/MyDrive/subprobe/probe_results"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

MIN_FREQ = 20

BOOTSTRAP_SAMPLES = 5000

SEEDS = [
    42,
    43,
    44,
    45,
    46
]

# =========================================================
# SET SEED
# =========================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

# =========================================================
# COPY PT FILES LOCALLY
# =========================================================

os.makedirs(
    LOCAL_REP_ROOT,
    exist_ok=True
)

pt_files = glob.glob(
    os.path.join(REP_ROOT, "*.pt")
)

print(f"Found {len(pt_files)} pt files")

for src_path in tqdm(pt_files):

    filename = os.path.basename(src_path)

    dst_path = os.path.join(
        LOCAL_REP_ROOT,
        filename
    )

    if not os.path.exists(dst_path):

        shutil.copy2(
            src_path,
            dst_path
        )

print("Finished local copy.")

# =========================================================
# LOAD CSV
# =========================================================

df = pd.read_csv(CSV_PATH)

df = df[[
    "id",
    "label_counts"
]]

# =========================================================
# REMOVE MISSING REPRESENTATIONS
# =========================================================

df = df[
    df["id"].apply(
        lambda x:
        os.path.exists(
            os.path.join(
                LOCAL_REP_ROOT,
                f"{x}.pt"
            )
        )
    )
].reset_index(drop=True)

print(f"Remaining samples: {len(df)}")

# =========================================================
# BUILD EMOTION VOCAB
# =========================================================

from collections import Counter

emotion_counter = Counter()

for counts_str in df["label_counts"]:

    counts = ast.literal_eval(counts_str)

    for emo, cnt in counts.items():

        if emo == "unsure":
            continue

        emotion_counter[emo] += cnt

EMOTIONS = sorted([

    emo

    for emo, freq
    in emotion_counter.items()

    if freq >= MIN_FREQ
])

EMO2IDX = {

    emo: i

    for i, emo
    in enumerate(EMOTIONS)
}

IDX2EMO = {

    i: emo

    for emo, i
    in EMO2IDX.items()
}

NUM_LABELS = len(EMOTIONS)

print("\nDetected emotions:")
print(EMOTIONS)

print(f"\nNUM_LABELS = {NUM_LABELS}")

# =========================================================
# BUILD TARGETS
# =========================================================

targets = []

valid_rows = []

for idx, row in df.iterrows():

    counts = ast.literal_eval(
        row["label_counts"]
    )

    counts = {

        k: v

        for k, v
        in counts.items()

        if k != "unsure"
    }

    if len(counts) == 0:
        continue

    y = np.zeros(NUM_LABELS)

    total = sum(counts.values())

    for emo, cnt in counts.items():

        if emo in EMO2IDX:

            y[EMO2IDX[emo]] = cnt / total

    targets.append(y)

    valid_rows.append(idx)

df = df.iloc[valid_rows].reset_index(drop=True)

targets = np.array(targets)

print("\nTargets shape:")
print(targets.shape)

# =========================================================
# STRATIFIED SPLITS
# =========================================================

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_idx, temp_idx = next(

    msss.split(
        np.zeros(len(targets)),
        (targets > 0).astype(int)
    )
)

temp_targets = targets[temp_idx]

msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.50,
    random_state=42
)

val_subidx, test_subidx = next(

    msss2.split(
        np.zeros(len(temp_targets)),
        (temp_targets > 0).astype(int)
    )
)

val_idx = temp_idx[val_subidx]

test_idx = temp_idx[test_subidx]

print("\nSplit sizes:")
print(
    len(train_idx),
    len(val_idx),
    len(test_idx)
)

# =========================================================
# SAVE SPLITS
# =========================================================

split_dict = {

    "train": train_idx.tolist(),

    "val": val_idx.tolist(),

    "test": test_idx.tolist()
}

with open(
    os.path.join(
        SAVE_DIR,
        "splits.json"
    ),
    "w"
) as f:

    json.dump(split_dict, f)

# =========================================================
# REPRESENTATION PROCESSING
# =========================================================

def process_rep(x):

    x = x.float()

    # -----------------------------------------------------
    # TOKEN POOLING
    # -----------------------------------------------------

    if x.dim() == 3:

        x = x.mean(dim=1)

    x = x.squeeze()

    if x.dim() == 2:

        x = x[0]

    # -----------------------------------------------------
    # NORMALIZATION
    # -----------------------------------------------------

    x = (

        x - x.mean()

    ) / (

        x.std() + 1e-6
    )

    return x

# =========================================================
# DATASET
# =========================================================

class CachedDataset(Dataset):

    def __init__(
        self,
        X,
        Y,
        ids
    ):

        self.X = X
        self.Y = Y
        self.ids = ids

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (

            self.X[idx],

            self.Y[idx],

            self.ids[idx]
        )

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(
        self,
        input_dim,
        num_labels
    ):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            num_labels
        )

    def forward(self, x):

        return self.linear(x)

# =========================================================
# THRESHOLD SEARCH
# =========================================================

def find_best_threshold(
    probs,
    labels
):

    best_thresh = 0.5

    best_f1 = -1

    thresholds = np.arange(
        0.05,
        0.95,
        0.05
    )

    for thresh in thresholds:

        preds = (
            probs > thresh
        ).astype(int)

        score = f1_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score

            best_thresh = thresh

    return best_thresh

# =========================================================
# EVALUATION
# =========================================================

def evaluate_full(
    model,
    loader,
    threshold=None
):

    model.eval()

    all_probs = []

    all_labels = []

    all_ids = []

    with torch.no_grad():

        for x, y, ids in loader:

            x = x.to(DEVICE)

            logits = model(x)

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                (y > 0).float().cpu().numpy()
            )

            all_ids.extend(ids)

    all_probs = np.concatenate(all_probs)

    all_labels = np.concatenate(all_labels)

    # -----------------------------------------------------
    # THRESHOLD TUNING
    # -----------------------------------------------------

    if threshold is None:

        threshold = find_best_threshold(
            all_probs,
            all_labels
        )

    all_preds = (
        all_probs > threshold
    ).astype(int)

    macro_f1 = f1_score(

        all_labels,

        all_preds,

        average="macro",

        zero_division=0
    )

    micro_f1 = f1_score(

        all_labels,

        all_preds,

        average="micro",

        zero_division=0
    )

    per_label_f1 = f1_score(

        all_labels,

        all_preds,

        average=None,

        zero_division=0
    )

    return {

        "macro_f1": macro_f1,

        "micro_f1": micro_f1,

        "per_label_f1": per_label_f1,

        "preds": all_preds,

        "labels": all_labels,

        "probs": all_probs,

        "threshold": threshold,

        "ids": all_ids
    }

# =========================================================
# BOOTSTRAP F1
# =========================================================

def bootstrap_f1(
    labels,
    preds,
    n_boot=5000
):

    scores = []

    n = len(labels)

    for _ in range(n_boot):

        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        sample_labels = labels[idx]

        sample_preds = preds[idx]

        score = f1_score(

            sample_labels,

            sample_preds,

            average="macro",

            zero_division=0
        )

        scores.append(score)

    scores = np.array(scores)

    return (

        scores.mean(),

        np.percentile(scores, 2.5),

        np.percentile(scores, 97.5)
    )

# =========================================================
# LAYERS
# =========================================================

SELECTED_LAYERS = [

    "audio_32",

    "projector",

    "lm_13",

    "lm_27"
]

# =========================================================
# MAIN LOOP
# =========================================================

all_results = []

all_predictions = []
baseline_outputs = {}
for layer_key in SELECTED_LAYERS:

    print("\n================================")
    print(layer_key)
    print("================================")

    # =====================================================
    # CACHE LAYER
    # =====================================================

    cached_X = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df)
    ):

        rep_id = row["id"]

        rep_path = os.path.join(
            LOCAL_REP_ROOT,
            f"{rep_id}.pt"
        )

        rep = torch.load(
            rep_path,
            map_location="cpu"
        )

        x = process_rep(
            rep[layer_key]
        )

        cached_X.append(x)

    cached_X = torch.stack(cached_X)

    cached_Y = torch.tensor(
        targets,
        dtype=torch.float
    )

    # =====================================================
    # MULTI-SEED
    # =====================================================

    for seed in SEEDS:

        print(f"\nSEED {seed}")

        set_seed(seed)

        train_dataset = CachedDataset(

            cached_X[train_idx],

            cached_Y[train_idx],

            df.iloc[train_idx]["id"].tolist()
        )

        val_dataset = CachedDataset(

            cached_X[val_idx],

            cached_Y[val_idx],

            df.iloc[val_idx]["id"].tolist()
        )

        test_dataset = CachedDataset(

            cached_X[test_idx],

            cached_Y[test_idx],

            df.iloc[test_idx]["id"].tolist()
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        model = LinearProbe(

            cached_X.shape[-1],

            NUM_LABELS

        ).to(DEVICE)

        criterion = nn.BCEWithLogitsLoss()

        optimizer = optim.Adam(
            model.parameters(),
            lr=LR
        )

        best_val = -1

        best_path = os.path.join(
            SAVE_DIR,
            f"{layer_key}_{seed}.pt"
        )

        best_threshold = 0.5

        # =================================================
        # TRAIN
        # =================================================

        for epoch in range(EPOCHS):

            model.train()

            for x, y, _ in train_loader:

                x = x.to(DEVICE)

                y = y.to(DEVICE)

                logits = model(x)

                loss = criterion(
                    logits,
                    y
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

            val_outputs = evaluate_full(
                model,
                val_loader
            )

            val_macro = val_outputs["macro_f1"]

            print(
                f"Epoch {epoch} | "
                f"Val Macro {val_macro:.4f}"
            )

            if val_macro > best_val:

                best_val = val_macro

                best_threshold = val_outputs["threshold"]

                torch.save(
                    model.state_dict(),
                    best_path
                )

        # =================================================
        # TEST
        # =================================================

        model.load_state_dict(
            torch.load(best_path)
        )

        outputs = evaluate_full(
            model,
            test_loader,
            threshold=best_threshold
        )
        baseline_outputs[seed] = {
            "labels": outputs["labels"],
            "preds": outputs["preds"]
        }

        mean_f1, ci_low, ci_high = bootstrap_f1(

            outputs["labels"],

            outputs["preds"],

            n_boot=BOOTSTRAP_SAMPLES
        )

        print(
            f"Macro F1: "
            f"{outputs['macro_f1']:.4f}"
        )

        print(
            f"95% CI: "
            f"[{ci_low:.4f}, {ci_high:.4f}]"
        )

        # -------------------------------------------------
        # SAVE RESULTS
        # -------------------------------------------------

        all_results.append({

            "layer": layer_key,

            "seed": seed,

            "macro_f1": outputs["macro_f1"],

            "micro_f1": outputs["micro_f1"],

            "threshold": best_threshold,

            "bootstrap_macro_f1": mean_f1,

            "ci_low": ci_low,

            "ci_high": ci_high
        })

        for emo_idx, emo_name in IDX2EMO.items():

            all_results.append({

                "layer": layer_key,

                "seed": seed,

                "emotion": emo_name,

                "per_emotion_f1":
                outputs["per_label_f1"][emo_idx]
            })

        for i in range(len(outputs["ids"])):

            all_predictions.append({

                "layer": layer_key,

                "seed": seed,

                "id": outputs["ids"][i],

                "probs": outputs["probs"][i],

                "preds": outputs["preds"][i]
            })

# =========================================================
# SAVE DATAFRAMES
# =========================================================

results_df = pd.DataFrame(all_results)

preds_df = pd.DataFrame(all_predictions)

results_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "direct_probe_results.csv"
    ),
    index=False
)

preds_df.to_pickle(
    os.path.join(
        SAVE_DIR,
        "direct_probe_predictions.pkl"
    )
)

print("\nSaved results.")

# =========================================================
# WILCOXON CONFIDENCE TEST
# =========================================================

layer_a = "projector"

layer_b = "lm_13"

# ---------------------------------------------------------
# USE BEST SEED ONLY
# ---------------------------------------------------------

best_seed_a = (

    results_df[
        results_df["layer"] == layer_a
    ]

    .sort_values(
        "macro_f1",
        ascending=False
    )

    .iloc[0]["seed"]
)

best_seed_b = (

    results_df[
        results_df["layer"] == layer_b
    ]

    .sort_values(
        "macro_f1",
        ascending=False
    )

    .iloc[0]["seed"]
)

a_df = preds_df[
    (preds_df["layer"] == layer_a)
    &
    (preds_df["seed"] == best_seed_a)
].sort_values("id")

b_df = preds_df[
    (preds_df["layer"] == layer_b)
    &
    (preds_df["seed"] == best_seed_b)
].sort_values("id")

a_probs = np.stack(
    a_df["probs"].values
)

b_probs = np.stack(
    b_df["probs"].values
)

a_conf = a_probs.max(axis=1)

b_conf = b_probs.max(axis=1)

wilcox = wilcoxon(
    a_conf,
    b_conf
)

print("\nWILCOXON TEST")

print(
    f"p = {wilcox.pvalue:.6f}"
)

# =========================================================
# LAYER PERFORMANCE PLOT
# =========================================================

plot_df = (

    results_df

    .dropna(subset=["macro_f1"])

    .groupby("layer")

    ["macro_f1"]

    .mean()

    .reset_index()
)

plt.figure(figsize=(8,5))

plt.plot(

    plot_df["layer"],

    plot_df["macro_f1"],

    marker="o"
)

plt.ylabel("Macro F1")

plt.title(
    "Layer Probe Performance"
)

plt.grid(True)

plt.show()

In [ ]:
# =========================================================
# RESIDUAL REINJECTION PROBE
# ADD AUDIO INFORMATION BACK INTO LLM SPACE
# =========================================================

import os
import ast
import json
import random

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import (
    f1_score
)

from scipy.stats import (
    wilcoxon
)

import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

# =========================================================
# CONFIG
# =========================================================

CSV_PATH = (
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv"
)

REP_ROOT = (
    "/content/reps_local_flamingo"
)

SAVE_DIR = (
    "/content/drive/MyDrive/subprobe/reinjection_results"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

BOOTSTRAP_SAMPLES = 5000

SEEDS = [
    42,
    43,
    44,
    45,
    46
]

# =========================================================
# RESIDUAL WEIGHTS
# =========================================================

ALPHAS = [

    0.00,

    0.01,

    0.03,

    0.05,

    0.10,

    0.20
]

# =========================================================
# LAYERS
# =========================================================

AUDIO_LAYER = "projector"

LLM_LAYER = "lm_13"

# =========================================================
# SET SEED
# =========================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

# =========================================================
# LOAD CSV
# =========================================================

df = pd.read_csv(CSV_PATH)

df = df[[
    "id",
    "label_counts"
]]

# =========================================================
# LOAD SPLITS
# =========================================================

with open(
    "/content/drive/MyDrive/subprobe/probe_results/splits.json"
) as f:

    split_dict = json.load(f)

train_idx = split_dict["train"]

val_idx = split_dict["val"]

test_idx = split_dict["test"]

# =========================================================
# BUILD EMOTION VOCAB
# =========================================================

from collections import Counter

emotion_counter = Counter()

for counts_str in df["label_counts"]:

    counts = ast.literal_eval(counts_str)

    for emo, cnt in counts.items():

        if emo == "unsure":
            continue

        emotion_counter[emo] += cnt

EMOTIONS = sorted([

    emo

    for emo, freq
    in emotion_counter.items()

    if freq >= 20
])

EMO2IDX = {

    emo: i

    for i, emo
    in enumerate(EMOTIONS)
}

IDX2EMO = {

    i: emo

    for emo, i
    in EMO2IDX.items()
}

NUM_LABELS = len(EMOTIONS)

print(EMOTIONS)

# =========================================================
# BUILD TARGETS
# =========================================================

targets = []

valid_rows = []

for idx, row in df.iterrows():

    counts = ast.literal_eval(
        row["label_counts"]
    )

    counts = {

        k: v

        for k, v
        in counts.items()

        if k != "unsure"
    }

    if len(counts) == 0:
        continue

    y = np.zeros(NUM_LABELS)

    total = sum(counts.values())

    for emo, cnt in counts.items():

        if emo in EMO2IDX:

            y[EMO2IDX[emo]] = cnt / total

    targets.append(y)

    valid_rows.append(idx)

df = df.iloc[valid_rows].reset_index(drop=True)

targets = np.array(targets)

# =========================================================
# REPRESENTATION PROCESSING
# =========================================================

def process_rep(x):

    x = x.float()

    if x.dim() == 3:

        x = x.mean(dim=1)

    x = x.squeeze()

    if x.dim() == 2:

        x = x[0]

    x = (

        x - x.mean()

    ) / (

        x.std() + 1e-6
    )

    return x

# =========================================================
# CACHE REPRESENTATIONS
# =========================================================

print("\nCaching representations...")

cached_audio = []

cached_llm = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    rep_id = row["id"]

    rep_path = os.path.join(
        REP_ROOT,
        f"{rep_id}.pt"
    )

    rep = torch.load(
        rep_path,
        map_location="cpu"
    )

    audio_x = process_rep(
        rep[AUDIO_LAYER]
    )

    llm_x = process_rep(
        rep[LLM_LAYER]
    )

    cached_audio.append(audio_x)

    cached_llm.append(llm_x)

cached_audio = torch.stack(cached_audio)

cached_llm = torch.stack(cached_llm)

cached_Y = torch.tensor(
    targets,
    dtype=torch.float
)

print("\nAudio shape:")
print(cached_audio.shape)

print("\nLLM shape:")
print(cached_llm.shape)

# =========================================================
# DATASET
# =========================================================

class FusionDataset(Dataset):

    def __init__(
        self,
        X,
        Y,
        ids
    ):

        self.X = X
        self.Y = Y
        self.ids = ids

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (

            self.X[idx],

            self.Y[idx],

            self.ids[idx]
        )

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(
        self,
        input_dim,
        num_labels
    ):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            num_labels
        )

    def forward(self, x):

        return self.linear(x)

# =========================================================
# THRESHOLD SEARCH
# =========================================================

def find_best_threshold(
    probs,
    labels
):

    best_thresh = 0.5

    best_f1 = -1

    thresholds = np.arange(
        0.05,
        0.95,
        0.05
    )

    for thresh in thresholds:

        preds = (
            probs > thresh
        ).astype(int)

        score = f1_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score

            best_thresh = thresh

    return best_thresh

# =========================================================
# EVALUATION
# =========================================================

def evaluate_full(
    model,
    loader,
    threshold=None
):

    model.eval()

    all_probs = []

    all_labels = []

    all_ids = []

    with torch.no_grad():

        for x, y, ids in loader:

            x = x.to(DEVICE)

            logits = model(x)

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                (y > 0).float().cpu().numpy()
            )

            all_ids.extend(ids)

    all_probs = np.concatenate(all_probs)

    all_labels = np.concatenate(all_labels)

    if threshold is None:

        threshold = find_best_threshold(
            all_probs,
            all_labels
        )

    all_preds = (
        all_probs > threshold
    ).astype(int)

    macro_f1 = f1_score(

        all_labels,

        all_preds,

        average="macro",

        zero_division=0
    )

    micro_f1 = f1_score(

        all_labels,

        all_preds,

        average="micro",

        zero_division=0
    )

    per_label_f1 = f1_score(

        all_labels,

        all_preds,

        average=None,

        zero_division=0
    )

    return {

        "macro_f1": macro_f1,

        "micro_f1": micro_f1,

        "per_label_f1": per_label_f1,

        "preds": all_preds,

        "labels": all_labels,

        "probs": all_probs,

        "threshold": threshold,

        "ids": all_ids
    }

# =========================================================
# BOOTSTRAP
# =========================================================

def bootstrap_f1(
    labels,
    preds,
    n_boot=5000
):

    scores = []

    n = len(labels)

    for _ in range(n_boot):

        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        sample_labels = labels[idx]

        sample_preds = preds[idx]

        score = f1_score(

            sample_labels,

            sample_preds,

            average="macro",

            zero_division=0
        )

        scores.append(score)

    scores = np.array(scores)

    return (

        scores.mean(),

        np.percentile(scores, 2.5),

        np.percentile(scores, 97.5)
    )

# =========================================================
# MAIN LOOP
# =========================================================

all_results = []

all_predictions = []

for alpha in ALPHAS:

    print("\n================================")
    print(f"ALPHA = {alpha}")
    print("================================")

    # =====================================================
    # RESIDUAL REINJECTION
    # =====================================================

    fused_X = (

        cached_llm

        +

        alpha * cached_audio
    )

    # =====================================================
    # MULTI-SEED
    # =====================================================

    for seed in SEEDS:

        print(f"\nSEED {seed}")

        set_seed(seed)

        train_dataset = FusionDataset(

            fused_X[train_idx],

            cached_Y[train_idx],

            df.iloc[train_idx]["id"].tolist()
        )

        val_dataset = FusionDataset(

            fused_X[val_idx],

            cached_Y[val_idx],

            df.iloc[val_idx]["id"].tolist()
        )

        test_dataset = FusionDataset(

            fused_X[test_idx],

            cached_Y[test_idx],

            df.iloc[test_idx]["id"].tolist()
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        model = LinearProbe(

            fused_X.shape[-1],

            NUM_LABELS

        ).to(DEVICE)

        criterion = nn.BCEWithLogitsLoss()

        optimizer = optim.Adam(
            model.parameters(),
            lr=LR
        )

        best_val = -1

        best_threshold = 0.5

        best_path = os.path.join(
            SAVE_DIR,
            f"alpha_{alpha}_seed_{seed}.pt"
        )

        # =================================================
        # TRAIN
        # =================================================

        for epoch in range(EPOCHS):

            model.train()

            for x, y, _ in train_loader:

                x = x.to(DEVICE)

                y = y.to(DEVICE)

                logits = model(x)

                loss = criterion(
                    logits,
                    y
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

            val_outputs = evaluate_full(
                model,
                val_loader
            )

            val_macro = val_outputs["macro_f1"]

            print(
                f"Epoch {epoch} | "
                f"Val Macro {val_macro:.4f}"
            )

            if val_macro > best_val:

                best_val = val_macro

                best_threshold = val_outputs["threshold"]

                torch.save(
                    model.state_dict(),
                    best_path
                )

        # =================================================
        # TEST
        # =================================================

        model.load_state_dict(
            torch.load(best_path)
        )

        outputs = evaluate_full(
            model,
            test_loader,
            threshold=best_threshold
        )

        mean_f1, ci_low, ci_high = bootstrap_f1(

            outputs["labels"],

            outputs["preds"],

            n_boot=BOOTSTRAP_SAMPLES
        )

        print(
            f"Macro F1: "
            f"{outputs['macro_f1']:.4f}"
        )

        print(
            f"95% CI: "
            f"[{ci_low:.4f}, {ci_high:.4f}]"
        )

        # -------------------------------------------------
        # SAVE RESULTS
        # -------------------------------------------------

        all_results.append({

            "alpha": alpha,

            "seed": seed,

            "macro_f1": outputs["macro_f1"],

            "micro_f1": outputs["micro_f1"],

            "threshold": best_threshold,

            "bootstrap_macro_f1": mean_f1,

            "ci_low": ci_low,

            "ci_high": ci_high
        })

        for emo_idx, emo_name in IDX2EMO.items():

            all_results.append({

                "alpha": alpha,

                "seed": seed,

                "emotion": emo_name,

                "per_emotion_f1":
                outputs["per_label_f1"][emo_idx]
            })

        for i in range(len(outputs["ids"])):

            all_predictions.append({

                "alpha": alpha,

                "seed": seed,

                "id": outputs["ids"][i],

                "probs": outputs["probs"][i],

                "preds": outputs["preds"][i]
            })

# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(all_results)

preds_df = pd.DataFrame(all_predictions)

results_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "reinjection_results.csv"
    ),
    index=False
)

preds_df.to_pickle(
    os.path.join(
        SAVE_DIR,
        "reinjection_predictions.pkl"
    )
)

print("\nSaved results.")

# =========================================================
# BEST ALPHA
# =========================================================

summary_df = (

    results_df

    .dropna(subset=["macro_f1"])

    .groupby("alpha")

    ["macro_f1"]

    .mean()

    .reset_index()
)

best_alpha = summary_df.loc[
    summary_df["macro_f1"].idxmax()
]["alpha"]

print(f"\nBest alpha = {best_alpha}")

# =========================================================
# WILCOXON TEST
# =========================================================

baseline_seed = (

    results_df[
        results_df["alpha"] == 0.0
    ]

    .sort_values(
        "macro_f1",
        ascending=False
    )

    .iloc[0]["seed"]
)

best_seed = (

    results_df[
        results_df["alpha"] == best_alpha
    ]

    .sort_values(
        "macro_f1",
        ascending=False
    )

    .iloc[0]["seed"]
)

baseline_df = preds_df[
    (preds_df["alpha"] == 0.0)
    &
    (preds_df["seed"] == baseline_seed)
].sort_values("id")

best_df = preds_df[
    (preds_df["alpha"] == best_alpha)
    &
    (preds_df["seed"] == best_seed)
].sort_values("id")

baseline_probs = np.stack(
    baseline_df["probs"].values
)

best_probs = np.stack(
    best_df["probs"].values
)

baseline_conf = baseline_probs.max(axis=1)

best_conf = best_probs.max(axis=1)

wilcox = wilcoxon(
    baseline_conf,
    best_conf
)

print("\nWILCOXON TEST")

print(
    f"p = {wilcox.pvalue:.6f}"
)

# =========================================================
# PLOT
# =========================================================

plot_df = (

    results_df

    .dropna(subset=["macro_f1"])

    .groupby("alpha")

    ["macro_f1"]

    .mean()

    .reset_index()
)

plt.figure(figsize=(8,5))

plt.plot(

    plot_df["alpha"],

    plot_df["macro_f1"],

    marker="o"
)

plt.xlabel("Residual Reinjection Alpha")

plt.ylabel("Macro F1")

plt.title(
    "Residual Reinjection Performance"
)

plt.grid(True)

plt.show()

In [ ]:
# =========================================================
# RESIDUAL REINJECTION PROBE
# ADD AUDIO INFORMATION BACK INTO LLM SPACE
# =========================================================

import os
import ast
import json
import random

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import (
    f1_score
)

from scipy.stats import (
    wilcoxon
)

import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

# =========================================================
# CONFIG
# =========================================================

CSV_PATH = (
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv"
)

REP_ROOT = (
    "/content/reps_local_flamingo"
)

SAVE_DIR = (
    "/content/drive/MyDrive/subprobe/reinjection_results"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

BOOTSTRAP_SAMPLES = 5000

SEEDS = [
    42,
    43,
    44,
    45,
    46
]

# =========================================================
# RESIDUAL WEIGHTS
# =========================================================

ALPHAS = [

    0.00,

    0.01,

    0.03,

    0.05,

    0.10,

    0.20
]

# =========================================================
# LAYERS
# =========================================================

AUDIO_LAYER = "projector"

LLM_LAYER = "lm_13"

# =========================================================
# SET SEED
# =========================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

# =========================================================
# LOAD CSV
# =========================================================

df = pd.read_csv(CSV_PATH)

df = df[[
    "id",
    "label_counts"
]]

# =========================================================
# LOAD SPLITS
# =========================================================

with open(
    "/content/drive/MyDrive/subprobe/probe_results/splits.json"
) as f:

    split_dict = json.load(f)

train_idx = split_dict["train"]

val_idx = split_dict["val"]

test_idx = split_dict["test"]

# =========================================================
# BUILD EMOTION VOCAB
# =========================================================

from collections import Counter

emotion_counter = Counter()

for counts_str in df["label_counts"]:

    counts = ast.literal_eval(counts_str)

    for emo, cnt in counts.items():

        if emo == "unsure":
            continue

        emotion_counter[emo] += cnt

EMOTIONS = sorted([

    emo

    for emo, freq
    in emotion_counter.items()

    if freq >= 20
])

EMO2IDX = {

    emo: i

    for i, emo
    in enumerate(EMOTIONS)
}

IDX2EMO = {

    i: emo

    for emo, i
    in EMO2IDX.items()
}

NUM_LABELS = len(EMOTIONS)

print(EMOTIONS)

# =========================================================
# BUILD TARGETS
# =========================================================

targets = []

valid_rows = []

for idx, row in df.iterrows():

    counts = ast.literal_eval(
        row["label_counts"]
    )

    counts = {

        k: v

        for k, v
        in counts.items()

        if k != "unsure"
    }

    if len(counts) == 0:
        continue

    y = np.zeros(NUM_LABELS)

    total = sum(counts.values())

    for emo, cnt in counts.items():

        if emo in EMO2IDX:

            y[EMO2IDX[emo]] = cnt / total

    targets.append(y)

    valid_rows.append(idx)

df = df.iloc[valid_rows].reset_index(drop=True)

targets = np.array(targets)

# =========================================================
# REPRESENTATION PROCESSING
# =========================================================

def process_rep(x):

    x = x.float()

    if x.dim() == 3:

        x = x.mean(dim=1)

    x = x.squeeze()

    if x.dim() == 2:

        x = x[0]

    x = (

        x - x.mean()

    ) / (

        x.std() + 1e-6
    )

    return x

# =========================================================
# CACHE REPRESENTATIONS
# =========================================================

print("\nCaching representations...")

cached_audio = []

cached_llm = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    rep_id = row["id"]

    rep_path = os.path.join(
        REP_ROOT,
        f"{rep_id}.pt"
    )

    rep = torch.load(
        rep_path,
        map_location="cpu"
    )

    audio_x = process_rep(
        rep[AUDIO_LAYER]
    )

    llm_x = process_rep(
        rep[LLM_LAYER]
    )

    cached_audio.append(audio_x)

    cached_llm.append(llm_x)

cached_audio = torch.stack(cached_audio)

cached_llm = torch.stack(cached_llm)

cached_Y = torch.tensor(
    targets,
    dtype=torch.float
)

print("\nAudio shape:")
print(cached_audio.shape)

print("\nLLM shape:")
print(cached_llm.shape)

# =========================================================
# DATASET
# =========================================================

class FusionDataset(Dataset):

    def __init__(
        self,
        X,
        Y,
        ids
    ):

        self.X = X
        self.Y = Y
        self.ids = ids

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (

            self.X[idx],

            self.Y[idx],

            self.ids[idx]
        )

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(
        self,
        input_dim,
        num_labels
    ):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            num_labels
        )

    def forward(self, x):

        return self.linear(x)

# =========================================================
# THRESHOLD SEARCH
# =========================================================

def find_best_threshold(
    probs,
    labels
):

    best_thresh = 0.5

    best_f1 = -1

    thresholds = np.arange(
        0.05,
        0.95,
        0.05
    )

    for thresh in thresholds:

        preds = (
            probs > thresh
        ).astype(int)

        score = f1_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score

            best_thresh = thresh

    return best_thresh

# =========================================================
# EVALUATION
# =========================================================

def evaluate_full(
    model,
    loader,
    threshold=None
):

    model.eval()

    all_probs = []

    all_labels = []

    all_ids = []

    with torch.no_grad():

        for x, y, ids in loader:

            x = x.to(DEVICE)

            logits = model(x)

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                (y > 0).float().cpu().numpy()
            )

            all_ids.extend(ids)

    all_probs = np.concatenate(all_probs)

    all_labels = np.concatenate(all_labels)

    if threshold is None:

        threshold = find_best_threshold(
            all_probs,
            all_labels
        )

    all_preds = (
        all_probs > threshold
    ).astype(int)

    macro_f1 = f1_score(

        all_labels,

        all_preds,

        average="macro",

        zero_division=0
    )

    micro_f1 = f1_score(

        all_labels,

        all_preds,

        average="micro",

        zero_division=0
    )

    per_label_f1 = f1_score(

        all_labels,

        all_preds,

        average=None,

        zero_division=0
    )

    return {

        "macro_f1": macro_f1,

        "micro_f1": micro_f1,

        "per_label_f1": per_label_f1,

        "preds": all_preds,

        "labels": all_labels,

        "probs": all_probs,

        "threshold": threshold,

        "ids": all_ids
    }

# =========================================================
# BOOTSTRAP
# =========================================================

def bootstrap_f1(
    labels,
    preds,
    n_boot=5000
):

    scores = []

    n = len(labels)

    for _ in range(n_boot):

        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        sample_labels = labels[idx]

        sample_preds = preds[idx]

        score = f1_score(

            sample_labels,

            sample_preds,

            average="macro",

            zero_division=0
        )

        scores.append(score)

    scores = np.array(scores)

    return (

        scores.mean(),

        np.percentile(scores, 2.5),

        np.percentile(scores, 97.5)
    )

# =========================================================
# MAIN LOOP
# =========================================================

all_results = []

all_predictions = []

for alpha in ALPHAS:

    print("\n================================")
    print(f"ALPHA = {alpha}")
    print("================================")

    # =====================================================
    # RESIDUAL REINJECTION
    # =====================================================

    fused_X = (

        cached_llm

        +

        alpha * cached_audio
    )

    # =====================================================
    # MULTI-SEED
    # =====================================================

    for seed in SEEDS:

        print(f"\nSEED {seed}")

        set_seed(seed)

        train_dataset = FusionDataset(

            fused_X[train_idx],

            cached_Y[train_idx],

            df.iloc[train_idx]["id"].tolist()
        )

        val_dataset = FusionDataset(

            fused_X[val_idx],

            cached_Y[val_idx],

            df.iloc[val_idx]["id"].tolist()
        )

        test_dataset = FusionDataset(

            fused_X[test_idx],

            cached_Y[test_idx],

            df.iloc[test_idx]["id"].tolist()
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        model = LinearProbe(

            fused_X.shape[-1],

            NUM_LABELS

        ).to(DEVICE)

        criterion = nn.BCEWithLogitsLoss()

        optimizer = optim.Adam(
            model.parameters(),
            lr=LR
        )

        best_val = -1

        best_threshold = 0.5

        best_path = os.path.join(
            SAVE_DIR,
            f"alpha_{alpha}_seed_{seed}.pt"
        )

        # =================================================
        # TRAIN
        # =================================================

        for epoch in range(EPOCHS):

            model.train()

            for x, y, _ in train_loader:

                x = x.to(DEVICE)

                y = y.to(DEVICE)

                logits = model(x)

                loss = criterion(
                    logits,
                    y
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

            val_outputs = evaluate_full(
                model,
                val_loader
            )

            val_macro = val_outputs["macro_f1"]

            print(
                f"Epoch {epoch} | "
                f"Val Macro {val_macro:.4f}"
            )

            if val_macro > best_val:

                best_val = val_macro

                best_threshold = val_outputs["threshold"]

                torch.save(
                    model.state_dict(),
                    best_path
                )

        # =================================================
        # TEST
        # =================================================

        model.load_state_dict(
            torch.load(best_path)
        )

        outputs = evaluate_full(
            model,
            test_loader,
            threshold=best_threshold
        )

        mean_f1, ci_low, ci_high = bootstrap_f1(

            outputs["labels"],

            outputs["preds"],

            n_boot=BOOTSTRAP_SAMPLES
        )

        print(
            f"Macro F1: "
            f"{outputs['macro_f1']:.4f}"
        )

        print(
            f"95% CI: "
            f"[{ci_low:.4f}, {ci_high:.4f}]"
        )

        # -------------------------------------------------
        # SAVE RESULTS
        # -------------------------------------------------

        all_results.append({

            "alpha": alpha,

            "seed": seed,

            "macro_f1": outputs["macro_f1"],

            "micro_f1": outputs["micro_f1"],

            "threshold": best_threshold,

            "bootstrap_macro_f1": mean_f1,

            "ci_low": ci_low,

            "ci_high": ci_high
        })

        for emo_idx, emo_name in IDX2EMO.items():

            all_results.append({

                "alpha": alpha,

                "seed": seed,

                "emotion": emo_name,

                "per_emotion_f1":
                outputs["per_label_f1"][emo_idx]
            })

        for i in range(len(outputs["ids"])):

            all_predictions.append({

                "alpha": alpha,

                "seed": seed,

                "id": outputs["ids"][i],

                "probs": outputs["probs"][i],

                "preds": outputs["preds"][i]
            })

# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(all_results)

preds_df = pd.DataFrame(all_predictions)

results_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "reinjection_results.csv"
    ),
    index=False
)

preds_df.to_pickle(
    os.path.join(
        SAVE_DIR,
        "reinjection_predictions.pkl"
    )
)

print("\nSaved results.")

# =========================================================
# BEST ALPHA
# =========================================================

summary_df = (

    results_df

    .dropna(subset=["macro_f1"])

    .groupby("alpha")

    ["macro_f1"]

    .mean()

    .reset_index()
)

best_alpha = summary_df.loc[
    summary_df["macro_f1"].idxmax()
]["alpha"]

print(f"\nBest alpha = {best_alpha}")

# =========================================================
# WILCOXON TEST
# =========================================================

baseline_seed = (

    results_df[
        results_df["alpha"] == 0.0
    ]

    .sort_values(
        "macro_f1",
        ascending=False
    )

    .iloc[0]["seed"]
)

best_seed = (

    results_df[
        results_df["alpha"] == best_alpha
    ]

    .sort_values(
        "macro_f1",
        ascending=False
    )

    .iloc[0]["seed"]
)

baseline_df = preds_df[
    (preds_df["alpha"] == 0.0)
    &
    (preds_df["seed"] == baseline_seed)
].sort_values("id")

best_df = preds_df[
    (preds_df["alpha"] == best_alpha)
    &
    (preds_df["seed"] == best_seed)
].sort_values("id")

baseline_probs = np.stack(
    baseline_df["probs"].values
)

best_probs = np.stack(
    best_df["probs"].values
)

baseline_conf = baseline_probs.max(axis=1)

best_conf = best_probs.max(axis=1)

wilcox = wilcoxon(
    baseline_conf,
    best_conf
)

print("\nWILCOXON TEST")

print(
    f"p = {wilcox.pvalue:.6f}"
)

# =========================================================
# PLOT
# =========================================================

plot_df = (

    results_df

    .dropna(subset=["macro_f1"])

    .groupby("alpha")

    ["macro_f1"]

    .mean()

    .reset_index()
)

plt.figure(figsize=(8,5))

plt.plot(

    plot_df["alpha"],

    plot_df["macro_f1"],

    marker="o"
)

plt.xlabel("Residual Reinjection Alpha")

plt.ylabel("Macro F1")

plt.title(
    "Residual Reinjection Performance"
)

plt.grid(True)

plt.show()

In [ ]:
# =========================================================
# RESIDUAL REINJECTION PROBE
# ADD AUDIO INFORMATION BACK INTO LLM SPACE
# =========================================================

import os
import ast
import json
import random

import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.metrics import (
    f1_score
)

from scipy.stats import (
    wilcoxon
)

import matplotlib.pyplot as plt

from torch.utils.data import (
    Dataset,
    DataLoader
)

# =========================================================
# CONFIG
# =========================================================

CSV_PATH = (
    "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/task3_results.csv"
)

REP_ROOT = (
    "/content/reps_local_flamingo"
)

SAVE_DIR = (
    "/content/drive/MyDrive/subprobe/reinjection_results2"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

BOOTSTRAP_SAMPLES = 5000

SEEDS = [
    42,
    43,
    44,
    45,
    46
]

# =========================================================
# RESIDUAL WEIGHTS
# =========================================================

ALPHAS = [

    0.00,

    0.01,

    0.03,

    0.05,

    0.10,

    0.20
]

# =========================================================
# LAYERS
# =========================================================

AUDIO_LAYER = "projector"

LLM_LAYER = "lm_2"

# =========================================================
# SET SEED
# =========================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

# =========================================================
# LOAD CSV
# =========================================================

df = pd.read_csv(CSV_PATH)

df = df[[
    "id",
    "label_counts"
]]

# =========================================================
# LOAD SPLITS
# =========================================================

with open(
    "/content/drive/MyDrive/subprobe/probe_results/splits.json"
) as f:

    split_dict = json.load(f)

train_idx = split_dict["train"]

val_idx = split_dict["val"]

test_idx = split_dict["test"]

# =========================================================
# BUILD EMOTION VOCAB
# =========================================================

from collections import Counter

emotion_counter = Counter()

for counts_str in df["label_counts"]:

    counts = ast.literal_eval(counts_str)

    for emo, cnt in counts.items():

        if emo == "unsure":
            continue

        emotion_counter[emo] += cnt

EMOTIONS = sorted([

    emo

    for emo, freq
    in emotion_counter.items()

    if freq >= 20
])

EMO2IDX = {

    emo: i

    for i, emo
    in enumerate(EMOTIONS)
}

IDX2EMO = {

    i: emo

    for emo, i
    in EMO2IDX.items()
}

NUM_LABELS = len(EMOTIONS)

print(EMOTIONS)

# =========================================================
# BUILD TARGETS
# =========================================================

targets = []

valid_rows = []

for idx, row in df.iterrows():

    counts = ast.literal_eval(
        row["label_counts"]
    )

    counts = {

        k: v

        for k, v
        in counts.items()

        if k != "unsure"
    }

    if len(counts) == 0:
        continue

    y = np.zeros(NUM_LABELS)

    total = sum(counts.values())

    for emo, cnt in counts.items():

        if emo in EMO2IDX:

            y[EMO2IDX[emo]] = cnt / total

    targets.append(y)

    valid_rows.append(idx)

df = df.iloc[valid_rows].reset_index(drop=True)

targets = np.array(targets)

# =========================================================
# REPRESENTATION PROCESSING
# =========================================================

def process_rep(x):

    x = x.float()

    if x.dim() == 3:

        x = x.mean(dim=1)

    x = x.squeeze()

    if x.dim() == 2:

        x = x[0]

    x = (

        x - x.mean()

    ) / (

        x.std() + 1e-6
    )

    return x

# =========================================================
# CACHE REPRESENTATIONS
# =========================================================

print("\nCaching representations...")

cached_audio = []

cached_llm = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df)
):

    rep_id = row["id"]

    rep_path = os.path.join(
        REP_ROOT,
        f"{rep_id}.pt"
    )

    rep = torch.load(
        rep_path,
        map_location="cpu"
    )

    audio_x = process_rep(
        rep[AUDIO_LAYER]
    )

    llm_x = process_rep(
        rep[LLM_LAYER]
    )

    cached_audio.append(audio_x)

    cached_llm.append(llm_x)

cached_audio = torch.stack(cached_audio)

cached_llm = torch.stack(cached_llm)

cached_Y = torch.tensor(
    targets,
    dtype=torch.float
)

print("\nAudio shape:")
print(cached_audio.shape)

print("\nLLM shape:")
print(cached_llm.shape)

# =========================================================
# DATASET
# =========================================================

class FusionDataset(Dataset):

    def __init__(
        self,
        X,
        Y,
        ids
    ):

        self.X = X
        self.Y = Y
        self.ids = ids

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (

            self.X[idx],

            self.Y[idx],

            self.ids[idx]
        )

# =========================================================
# MODEL
# =========================================================

class LinearProbe(nn.Module):

    def __init__(
        self,
        input_dim,
        num_labels
    ):

        super().__init__()

        self.linear = nn.Linear(
            input_dim,
            num_labels
        )

    def forward(self, x):

        return self.linear(x)

# =========================================================
# THRESHOLD SEARCH
# =========================================================

def find_best_threshold(
    probs,
    labels
):

    best_thresh = 0.5

    best_f1 = -1

    thresholds = np.arange(
        0.05,
        0.95,
        0.05
    )

    for thresh in thresholds:

        preds = (
            probs > thresh
        ).astype(int)

        score = f1_score(
            labels,
            preds,
            average="macro",
            zero_division=0
        )

        if score > best_f1:

            best_f1 = score

            best_thresh = thresh

    return best_thresh

# =========================================================
# EVALUATION
# =========================================================

def evaluate_full(
    model,
    loader,
    threshold=None
):

    model.eval()

    all_probs = []

    all_labels = []

    all_ids = []

    with torch.no_grad():

        for x, y, ids in loader:

            x = x.to(DEVICE)

            logits = model(x)

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                (y > 0).float().cpu().numpy()
            )

            all_ids.extend(ids)

    all_probs = np.concatenate(all_probs)

    all_labels = np.concatenate(all_labels)

    if threshold is None:

        threshold = find_best_threshold(
            all_probs,
            all_labels
        )

    all_preds = (
        all_probs > threshold
    ).astype(int)

    macro_f1 = f1_score(

        all_labels,

        all_preds,

        average="macro",

        zero_division=0
    )

    micro_f1 = f1_score(

        all_labels,

        all_preds,

        average="micro",

        zero_division=0
    )

    per_label_f1 = f1_score(

        all_labels,

        all_preds,

        average=None,

        zero_division=0
    )

    return {

        "macro_f1": macro_f1,

        "micro_f1": micro_f1,

        "per_label_f1": per_label_f1,

        "preds": all_preds,

        "labels": all_labels,

        "probs": all_probs,

        "threshold": threshold,

        "ids": all_ids
    }

# =========================================================
# BOOTSTRAP
# =========================================================

def bootstrap_f1(
    labels,
    preds,
    n_boot=5000
):

    scores = []

    n = len(labels)

    for _ in range(n_boot):

        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        score = f1_score(
            labels[idx],
            preds[idx],
            average="macro",
            zero_division=0
        )

        scores.append(score)

    scores = np.array(scores)

    return (
        scores.mean(),
        np.percentile(scores,2.5),
        np.percentile(scores,97.5)
    )


def bootstrap_difference(
    labels,
    preds_base,
    preds_alpha,
    n_boot=5000
):

    n = len(labels)

    diffs = []

    for _ in range(n_boot):

        idx = np.random.choice(
            n,
            n,
            replace=True
        )

        f1_base = f1_score(
            labels[idx],
            preds_base[idx],
            average="macro",
            zero_division=0
        )

        f1_alpha = f1_score(
            labels[idx],
            preds_alpha[idx],
            average="macro",
            zero_division=0
        )

        diffs.append(
            f1_alpha - f1_base
        )

    diffs = np.array(diffs)

    return (
        diffs.mean(),
        np.percentile(diffs,2.5),
        np.percentile(diffs,97.5)
    )

# =========================================================
# MAIN LOOP
# =========================================================

all_results = []

all_predictions = []
baseline_outputs = {}

seed_results = {
    alpha: []
    for alpha in ALPHAS
}
for alpha in ALPHAS:

    print("\n================================")
    print(f"ALPHA = {alpha}")
    print("================================")

    # =====================================================
    # RESIDUAL REINJECTION
    # =====================================================

    fused_X = (

        cached_llm

        +

        alpha * cached_audio
    )

    # =====================================================
    # MULTI-SEED
    # =====================================================

    for seed in SEEDS:

        print(f"\nSEED {seed}")

        set_seed(seed)

        train_dataset = FusionDataset(

            fused_X[train_idx],

            cached_Y[train_idx],

            df.iloc[train_idx]["id"].tolist()
        )

        val_dataset = FusionDataset(

            fused_X[val_idx],

            cached_Y[val_idx],

            df.iloc[val_idx]["id"].tolist()
        )

        test_dataset = FusionDataset(

            fused_X[test_idx],

            cached_Y[test_idx],

            df.iloc[test_idx]["id"].tolist()
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        test_loader = DataLoader(
            test_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False
        )

        model = LinearProbe(

            fused_X.shape[-1],

            NUM_LABELS

        ).to(DEVICE)

        criterion = nn.BCEWithLogitsLoss()

        optimizer = optim.Adam(
            model.parameters(),
            lr=LR
        )

        best_val = -1

        best_threshold = 0.5

        best_path = os.path.join(
            SAVE_DIR,
            f"alpha_{alpha}_seed_{seed}.pt"
        )

        # =================================================
        # TRAIN
        # =================================================

        for epoch in range(EPOCHS):

            model.train()

            for x, y, _ in train_loader:

                x = x.to(DEVICE)

                y = y.to(DEVICE)

                logits = model(x)

                loss = criterion(
                    logits,
                    y
                )

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

            val_outputs = evaluate_full(
                model,
                val_loader
            )

            val_macro = val_outputs["macro_f1"]

            print(
                f"Epoch {epoch} | "
                f"Val Macro {val_macro:.4f}"
            )

            if val_macro > best_val:

                best_val = val_macro

                best_threshold = val_outputs["threshold"]

                torch.save(
                    model.state_dict(),
                    best_path
                )

        # =================================================
        # TEST
        # =================================================

        model.load_state_dict(
            torch.load(best_path)
        )

        outputs = evaluate_full(
            model,
            test_loader,
            threshold=best_threshold
        )
        seed_results[alpha].append(
            outputs["macro_f1"]
        )

        if alpha == 0:

            mean_f1, ci_low, ci_high = bootstrap_f1(
                outputs["labels"],
                outputs["preds"],
                BOOTSTRAP_SAMPLES
            )

            baseline_outputs[seed] = {

                "labels": outputs["labels"],

                "preds": outputs["preds"]

            }

            print(
                f"Macro F1: {outputs['macro_f1']:.4f}"
            )

            print(
                f"95% CI: [{ci_low:.4f}, {ci_high:.4f}]"
            )

            diff_mean = 0
            diff_low = 0
            diff_high = 0

        else:

            diff_mean, diff_low, diff_high = bootstrap_difference(

                baseline_outputs[seed]["labels"],

                baseline_outputs[seed]["preds"],

                outputs["preds"],

                BOOTSTRAP_SAMPLES

            )

            print(
                f"Macro F1: {outputs['macro_f1']:.4f}"
            )

            print(
                f"Δ vs α=0 : {diff_mean:+.4f}"
            )

            print(
                f"95% CI : [{diff_low:+.4f}, {diff_high:+.4f}]"
            )

        # -------------------------------------------------
        # SAVE RESULTS
        # -------------------------------------------------

        all_results.append({

            "alpha": alpha,

            "seed": seed,

            "macro_f1": outputs["macro_f1"],

            "micro_f1": outputs["micro_f1"],

            "threshold": best_threshold,

            "bootstrap_macro_f1": mean_f1 if alpha==0 else np.nan,

            "ci_low": ci_low if alpha==0 else np.nan,

            "ci_high": ci_high if alpha==0 else np.nan,

            "delta_macro": diff_mean,

            "delta_ci_low": diff_low,

            "delta_ci_high": diff_high

        })

        for emo_idx, emo_name in IDX2EMO.items():

            all_results.append({

                "alpha": alpha,

                "seed": seed,

                "emotion": emo_name,

                "per_emotion_f1":
                outputs["per_label_f1"][emo_idx]
            })

        for i in range(len(outputs["ids"])):

            all_predictions.append({

                "alpha": alpha,

                "seed": seed,

                "id": outputs["ids"][i],

                "probs": outputs["probs"][i],

                "preds": outputs["preds"][i]
            })
print("\n===================================")
print("PAIRED SEED COMPARISON")
print("===================================")

baseline = np.array(seed_results[0.0])

for alpha in ALPHAS:

    if alpha == 0:
        continue

    scores = np.array(
        seed_results[alpha]
    )

    delta = scores - baseline

    print()

    print(f"Alpha = {alpha}")

    print(
        f"Mean Δ = {delta.mean():+.4f}"
    )

    print(
        f"Std Δ = {delta.std(ddof=1):.4f}"
    )

    stat, p = wilcoxon(
        delta,
        alternative="two-sided"
    )

    print(
        f"Wilcoxon p = {p:.4f}"
    )
# =========================================================
# SAVE RESULTS
# =========================================================

results_df = pd.DataFrame(all_results)

preds_df = pd.DataFrame(all_predictions)

results_df.to_csv(
    os.path.join(
        SAVE_DIR,
        "reinjection_results.csv"
    ),
    index=False
)

preds_df.to_pickle(
    os.path.join(
        SAVE_DIR,
        "reinjection_predictions.pkl"
    )
)

print("\nSaved results.")

# =========================================================
# BEST ALPHA
# =========================================================

summary_df = (

    results_df

    .dropna(subset=["macro_f1"])

    .groupby("alpha")

    ["macro_f1"]

    .mean()

    .reset_index()
)

best_alpha = summary_df.loc[
    summary_df["macro_f1"].idxmax()
]["alpha"]

print(f"\nBest alpha = {best_alpha}")

# =========================================================
# WILCOXON TEST
# =========================================================

baseline_seed = (

    results_df[
        results_df["alpha"] == 0.0
    ]

    .sort_values(
        "macro_f1",
        ascending=False
    )

    .iloc[0]["seed"]
)

best_seed = (

    results_df[
        results_df["alpha"] == best_alpha
    ]

    .sort_values(
        "macro_f1",
        ascending=False
    )

    .iloc[0]["seed"]
)

baseline_df = preds_df[
    (preds_df["alpha"] == 0.0)
    &
    (preds_df["seed"] == baseline_seed)
].sort_values("id")

best_df = preds_df[
    (preds_df["alpha"] == best_alpha)
    &
    (preds_df["seed"] == best_seed)
].sort_values("id")

baseline_probs = np.stack(
    baseline_df["probs"].values
)

best_probs = np.stack(
    best_df["probs"].values
)

baseline_conf = baseline_probs.max(axis=1)

best_conf = best_probs.max(axis=1)

wilcox = wilcoxon(
    baseline_conf,
    best_conf
)

print("\nWILCOXON TEST")

print(
    f"p = {wilcox.pvalue:.6f}"
)

# =========================================================
# PLOT
# =========================================================

plot_df = (

    results_df

    .dropna(subset=["macro_f1"])

    .groupby("alpha")

    ["macro_f1"]

    .mean()

    .reset_index()
)

plt.figure(figsize=(8,5))

plt.plot(

    plot_df["alpha"],

    plot_df["macro_f1"],

    marker="o"
)

plt.xlabel("Residual Reinjection Alpha")

plt.ylabel("Macro F1")

plt.title(
    "Residual Reinjection Performance"
)

plt.grid(True)

plt.show()